In [3]:
# Import libraries
import pandas as pd
import numpy as np
from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.core.metrics import make_scorer

## 1. Load Data

In [4]:
# Load sample dataset
print("="*80)
print("Loading dataset...")
print("="*80)

# Using Titanic dataset as example
train_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/Inc/train.csv')
test_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/Inc/test.csv')

print(f"Train data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")
print("\nFirst 5 rows:")
display(train_data.head())

# Check label column
label = 'class'
print(f"\nTarget variable: {label}")
print(f"Classes: {train_data[label].unique()}")

Loading dataset...
Train data shape: (39073, 15)
Test data shape: (9769, 15)

First 5 rows:


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,class
0,25,Private,178478,Bachelors,13,Never-married,Tech-support,Own-child,White,Female,0,0,40,United-States,<=50K
1,23,State-gov,61743,5th-6th,3,Never-married,Transport-moving,Not-in-family,White,Male,0,0,35,United-States,<=50K
2,46,Private,376789,HS-grad,9,Never-married,Other-service,Not-in-family,White,Male,0,0,15,United-States,<=50K
3,55,?,200235,HS-grad,9,Married-civ-spouse,?,Husband,White,Male,0,0,50,United-States,>50K
4,36,Private,224541,7th-8th,4,Married-civ-spouse,Handlers-cleaners,Husband,White,Male,0,0,40,El-Salvador,<=50K



Target variable: class
Classes: [' <=50K' ' >50K']


## 2. Basic Training

In [5]:
# Train with default settings (Quick)
print("\n" + "="*80)
print("Training AutoGluon with default settings...")
print("="*80)

predictor = TabularPredictor(
    label=label,
    problem_type='binary',  # or 'multiclass', 'regression'
    eval_metric='accuracy',
    path='./autogluon_models'
).fit(
    train_data=train_data,
    time_limit=120,  # seconds
    presets='medium_quality'  # 'best_quality', 'high_quality', 'good_quality', 'medium_quality'
)

print("\nTraining complete!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.18
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Aug 11 21:16:21 PDT 2025; root:xnu-11417.140.69.701.11~1/RELEASE_ARM64_T6000
CPU Count:          10
Memory Avail:       9.50 GB / 32.00 GB (29.7%)
Disk Space Avail:   390.70 GB / 926.35 GB (42.2%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ... Time limit = 120s
AutoGluon will save models to "/Users/tarekatwan/Repos/MyWork/Teach/Ensemble Methods/Creo/demos/AutoML/AutoML_Patch_1/notebooks/autogluon_models"
Train Data Rows:    39073
Train Data Columns: 14
Label Column:       class
Problem Type:       binary
Preprocessing data ...
Selected class <--> label mapping:  class 1 =  >50K, class 0 =  <=50K
	Note: For your binary classification, AutoGluon arbitrarily selected wh


Training AutoGluon with default settings...


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 1 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('int', [])    : 6 | ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', ...]
		('object', []) : 8 | ['workclass', 'education', 'marital-status', 'occupation', 'relationship', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])  : 7 | ['workclass', 'education', 'marital-status', 'occupation', 'relationship', ...]
		('int', [])       : 6 | ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-los


Training complete!


## 3. Model Leaderboard

In [6]:
# View all trained models ranked by performance
print("\n" + "="*80)
print("Model Leaderboard:")
print("="*80)

leaderboard = predictor.leaderboard(test_data, silent=True)
display(leaderboard)


Model Leaderboard:


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGBoost,0.876139,0.8848,accuracy,0.076619,0.005959,0.740069,0.076619,0.005959,0.740069,1,True,8
1,WeightedEnsemble_L2,0.876139,0.8848,accuracy,0.078091,0.006355,0.776625,0.001472,0.000396,0.036556,2,True,10
2,LightGBMLarge,0.875422,0.8824,accuracy,0.033305,0.009485,6.990334,0.033305,0.009485,6.990334,1,True,9
3,CatBoost,0.875218,0.8828,accuracy,0.030412,0.004430,7.777475,0.030412,0.004430,7.777475,1,True,5
4,LightGBM,0.873477,0.8824,accuracy,0.032935,0.007714,2.624439,0.032935,0.007714,2.624439,1,True,2
5,LightGBMXT,0.871430,0.8792,accuracy,0.043929,0.011566,3.320400,0.043929,0.011566,3.320400,1,True,1
6,RandomForestGini,0.859760,0.8576,accuracy,0.159243,0.039861,1.147967,0.159243,0.039861,1.147967,1,True,3
7,RandomForestEntr,0.859249,0.8584,accuracy,0.159294,0.041547,1.178864,0.159294,0.041547,1.178864,1,True,4
8,ExtraTreesGini,0.851571,0.8484,accuracy,0.217237,0.041452,0.808438,0.217237,0.041452,0.808438,1,True,6
9,ExtraTreesEntr,0.850752,0.8508,accuracy,0.232598,0.038284,0.793237,0.232598,0.038284,0.793237,1,True,7


## 4. Best Model Info

In [11]:
# Get information about the best model
print("\n" + "="*80)
print("Best Model Information:")
print("="*80)

best_model = predictor.model_best
print(f"Best model: {best_model}")

model_info = predictor.info()
print(f"\nTotal models trained: {len(model_info['model_info'])}")


Best Model Information:
Best model: WeightedEnsemble_L2

Total models trained: 10


## 5. Predictions

In [12]:
# Make predictions
print("\n" + "="*80)
print("Making predictions...")
print("="*80)

y_pred = predictor.predict(test_data)
print("\nPredictions (first 10):")
display(y_pred.head(10))

# Get prediction probabilities
y_pred_proba = predictor.predict_proba(test_data)
print("\nPrediction Probabilities (first 5):")
display(y_pred_proba.head())


Making predictions...

Predictions (first 10):


0     <=50K
1     <=50K
2      >50K
3     <=50K
4     <=50K
5      >50K
6      >50K
7     <=50K
8     <=50K
9     <=50K
Name: class, dtype: object


Prediction Probabilities (first 5):


,<=50K,>50K
0,0.915430,0.084570
1,0.998447,0.001553
2,0.019262,0.980738
3,0.997421,0.002579
4,0.999018,0.000982


## 6. Model Evaluation

In [13]:
# Evaluate model performance
print("\n" + "="*80)
print("Model Evaluation:")
print("="*80)

performance = predictor.evaluate(test_data, silent=True)
print(f"Test Accuracy: {performance}")

# Detailed evaluation for each model
print("\nPer-model evaluation:")
for model_name in predictor.model_names():
    score = predictor.evaluate(test_data, model=model_name, silent=True)
    print(f"{model_name}: {score}")


Model Evaluation:
Test Accuracy: {'accuracy': 0.8761388064284983, 'balanced_accuracy': 0.797843987100538, 'mcc': 0.6402450906180814, 'roc_auc': 0.932099321549626, 'f1': 0.7131341868183974, 'precision': 0.791578947368421, 'recall': 0.6488352027610008}

Per-model evaluation:
LightGBMXT: {'accuracy': 0.8714300337803256, 'balanced_accuracy': 0.796688928494464, 'mcc': 0.6288949550553846, 'roc_auc': 0.9267165266916706, 'f1': 0.7072261072261072, 'precision': 0.7692697768762677, 'recall': 0.6544434857635893}
LightGBM: {'accuracy': 0.8734773262360528, 'balanced_accuracy': 0.7965450491673585, 'mcc': 0.6334410242076058, 'roc_auc': 0.9307796557294832, 'f1': 0.7091764705882353, 'precision': 0.7800207039337475, 'recall': 0.6501294219154443}
RandomForestGini: {'accuracy': 0.8597604667826799, 'balanced_accuracy': 0.7774482963703386, 'mcc': 0.5928803667203673, 'roc_auc': 0.9112494990278158, 'f1': 0.6774952919020716, 'precision': 0.7455958549222798, 'recall': 0.6207937877480587}
RandomForestEntr: {'acc

## 7. Feature Importance

In [14]:
# Analyze feature importance
print("\n" + "="*80)
print("Feature Importance:")
print("="*80)

feature_importance = predictor.feature_importance(test_data)
print(feature_importance)

Computing feature importance via permutation shuffling for 14 features using 5000 rows with 5 shuffle sets...
	4.01s	= Expected runtime (0.8s per shuffle set)



Feature Importance:


	1.22s	= Actual runtime (Completed 5 of 5 shuffle sets)


                importance    stddev   p_value  n  p99_high   p99_low
marital-status     0.05164  0.003321  0.000002  5  0.058478  0.044802
capital-gain       0.04696  0.004853  0.000013  5  0.056952  0.036968
education-num      0.03180  0.005030  0.000073  5  0.042157  0.021443
age                0.01556  0.003477  0.000280  5  0.022719  0.008401
occupation         0.01456  0.002889  0.000177  5  0.020509  0.008611
capital-loss       0.01260  0.001631  0.000033  5  0.015958  0.009242
hours-per-week     0.00820  0.002135  0.000505  5  0.012597  0.003803
workclass          0.00304  0.001780  0.009396  5  0.006705 -0.000625
relationship       0.00212  0.001706  0.024961  5  0.005634 -0.001394
fnlwgt             0.00196  0.001499  0.021555  5  0.005047 -0.001127
education          0.00140  0.000616  0.003544  5  0.002669  0.000131
sex                0.00104  0.001307  0.074894  5  0.003731 -0.001651
race               0.00064  0.001889  0.245429  5  0.004529 -0.003249
native-country     0

## 8. Advanced Training

In [15]:
# Train with custom hyperparameters and model selection
print("\n" + "="*80)
print("Advanced Training with Custom Settings...")
print("="*80)

predictor_advanced = TabularPredictor(
    label=label,
    problem_type='binary',
    eval_metric='roc_auc',
    path='./autogluon_advanced'
).fit(
    train_data=train_data,
    time_limit=180,
    presets='best_quality',
    hyperparameters={
        'GBM': {},  # LightGBM
        'XGB': {},  # XGBoost
        'CAT': {},  # CatBoost
        'RF': {'n_estimators': 300},  # Random Forest
        'NN_TORCH': {},  # Neural Network
    },
    num_bag_folds=5,  # Number of bagging folds
    num_bag_sets=1,
    num_stack_levels=1  # Stacking levels
)

print("\nAdvanced training complete!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.18
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Aug 11 21:16:21 PDT 2025; root:xnu-11417.140.69.701.11~1/RELEASE_ARM64_T6000
CPU Count:          10
Memory Avail:       8.91 GB / 32.00 GB (27.8%)
Disk Space Avail:   390.68 GB / 926.35 GB (42.2%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of th


Advanced Training with Custom Settings...


Selected class <--> label mapping:  class 1 =  >50K, class 0 =  <=50K
	Note: For your binary classification, AutoGluon arbitrarily selected which label-value represents positive ( >50K) vs negative ( <=50K) class.
	To explicitly set the positive_class, either rename classes to 1 and 0, or specify positive_class in Predictor init.
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    9131.52 MB
	Train Data (Original)  Memory Usage: 19.43 MB (0.2% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 1 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting Catego


Advanced training complete!


## 9. Hyperparameter Tuning

In [16]:
# Fine-tune hyperparameters
print("\n" + "="*80)
print("Hyperparameter Tuning...")
print("="*80)

hyperparameter_tune_kwargs = {
    'num_trials': 5,
    'scheduler': 'local',
    'searcher': 'auto',
}

predictor_tuned = TabularPredictor(
    label=label,
    path='./autogluon_tuned'
).fit(
    train_data=train_data,
    time_limit=120,
    hyperparameter_tune_kwargs=hyperparameter_tune_kwargs
)

print("\nHyperparameter tuning complete!")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.10.18
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.6.0: Mon Aug 11 21:16:21 PDT 2025; root:xnu-11417.140.69.701.11~1/RELEASE_ARM64_T6000
CPU Count:          10
Memory Avail:       9.23 GB / 32.00 GB (28.8%)
Disk Space Avail:   390.23 GB / 926.35 GB (42.1%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU memory.
	presets='best'


Hyperparameter Tuning...


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 1 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('int', [])    : 6 | ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', ...]
		('object', []) : 8 | ['workclass', 'education', 'marital-status', 'occupation', 'relationship', ...]
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])  : 7 | ['workclass', 'education', 'marital-status', 'occupation', 'relationship', ...]
		('int', [])       : 6 | ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-los

  0%|          | 0/5 [00:00<?, ?it/s]

	Ran out of time, early stopping on iteration 243. Best iteration is:
	[213]	valid_set's binary_error: 0.1326
	Stopping HPO to satisfy time limit...
Fitted model: LightGBMXT/T1 ...
	0.8654	 = Validation score   (accuracy)
	1.33s	 = Training   runtime
	0.0s	 = Validation runtime
Fitted model: LightGBMXT/T2 ...
	0.8686	 = Validation score   (accuracy)
	4.51s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: LightGBMXT/T3 ...
	0.8674	 = Validation score   (accuracy)
	2.93s	 = Training   runtime
	0.02s	 = Validation runtime
Hyperparameter tuning model: LightGBM ... Tuning model for up to 9.8s of the 110.88s of remaining time.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`


...

  0%|          | 0/5 [00:00<?, ?it/s]

Fitted model: XGBoost/T1 ...
	0.8774	 = Validation score   (accuracy)
	0.9s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: XGBoost/T2 ...
	0.8784	 = Validation score   (accuracy)
	0.98s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: XGBoost/T3 ...
	0.8786	 = Validation score   (accuracy)
	2.14s	 = Training   runtime
	0.02s	 = Validation runtime
Fitted model: XGBoost/T4 ...
	0.8768	 = Validation score   (accuracy)
	1.0s	 = Training   runtime
	0.01s	 = Validation runtime
Fitted model: XGBoost/T5 ...
	0.8778	 = Validation score   (accuracy)
	0.88s	 = Training   runtime
	0.01s	 = Validation runtime
Hyperparameter tuning model: NeuralNetTorch ... Tuning model for up to 9.8s of the 80.16s of remaining time.
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch

  0%|          | 0/5 [00:00<?, ?it/s]

Unable to import dependency torch
A quick tip is to install via `pip install torch`.
The minimum torch version is currently 2.2.
Traceback (most recent call last):
  File "/Users/tarekatwan/Repos/MyWork/Teach/Ensemble Methods/Creo/demos/AutoML/dev1/lib/python3.10/site-packages/autogluon/common/utils/try_import.py", line 148, in try_import_torch
    import torch
ModuleNotFoundError: No module named 'torch'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/tarekatwan/Repos/MyWork/Teach/Ensemble Methods/Creo/demos/AutoML/dev1/lib/python3.10/site-packages/autogluon/core/models/abstract/model_trial.py", line 37, in model_trial
    model = fit_and_save_model(
  File "/Users/tarekatwan/Repos/MyWork/Teach/Ensemble Methods/Creo/demos/AutoML/dev1/lib/python3.10/site-packages/autogluon/core/models/abstract/model_trial.py", line 96, in fit_and_save_model
    model.fit(**fit_args, time_limit=time_left)
  File "/Users/tarekatwan/R


Hyperparameter tuning complete!


## 10. Custom Metrics

In [ ]:
# Define and use custom evaluation metrics
print("\n" + "="*80)
print("Using Custom Metrics...")
print("="*80)

from sklearn.metrics import f1_score

def custom_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='weighted')

custom_metric = make_scorer(
    name='custom_f1',
    score_func=custom_f1,
    optimum=1,
    greater_is_better=True
)

# Use custom metric in training
predictor_custom = TabularPredictor(
    label=label,
    eval_metric=custom_metric,
    path='./autogluon_custom'
).fit(
    train_data=train_data,
    time_limit=60
)

print("\nCustom metric training complete!")

## 11. Persist & Load Model

In [17]:
# Save and load models
print("\n" + "="*80)
print("Model Persistence...")
print("="*80)

# Models are automatically saved during training
# Load a saved model
predictor_loaded = TabularPredictor.load('./autogluon_models')
print("Model loaded successfully!")

# Make predictions with loaded model
predictions_loaded = predictor_loaded.predict(test_data.head(10))
print("\nPredictions from loaded model:")
print(predictions_loaded)


Model Persistence...
Model loaded successfully!

Predictions from loaded model:
0     <=50K
1     <=50K
2      >50K
3     <=50K
4     <=50K
5      >50K
6      >50K
7     <=50K
8     <=50K
9     <=50K
Name: class, dtype: object


## 12. Model Inference

In [18]:
# Optimize for inference speed
print("\n" + "="*80)
print("Model Inference Optimization...")
print("="*80)

# Get fastest model within accuracy threshold
leaderboard_full = predictor.leaderboard(test_data)
print("\nModel comparison (accuracy vs inference time):")
print(leaderboard_full[['model', 'score_test', 'pred_time_test']])


Model Inference Optimization...

Model comparison (accuracy vs inference time):
                 model  score_test  pred_time_test
0              XGBoost    0.876139        0.083351
1  WeightedEnsemble_L2    0.876139        0.085005
2        LightGBMLarge    0.875422        0.034235
3             CatBoost    0.875218        0.043427
4             LightGBM    0.873477        0.026568
5           LightGBMXT    0.871430        0.044450
6     RandomForestGini    0.859760        0.156376
7     RandomForestEntr    0.859249        0.150226
8       ExtraTreesGini    0.851571        0.212496
9       ExtraTreesEntr    0.850752        0.243287


## 13. Distillation

In [ ]:
# Create a distilled student model
print("\n" + "="*80)
print("Model Distillation...")
print("="*80)

# Distill ensemble into single model for faster inference
predictor.distill(time_limit=60, augment_method='munge')
print("\nDistillation complete!")

## 14. Refit Models

In [ ]:
# Refit models on full dataset
print("\n" + "="*80)
print("Refitting on full dataset...")
print("="*80)

# Combine train and test for final model
full_data = pd.concat([train_data, test_data], ignore_index=True)
predictor.refit_full(model='all')
print("\nRefit complete!")

## 15. Multi-Modal Support

In [21]:
# AutoGluon Capabilities Summary
print("\n" + "="*80)
print("AutoGluon Capabilities Summary:")
print("="*80)
print("✓ Tabular Data (TabularPredictor)")
print("✓ Image Classification (ImagePredictor)")
print("✓ Object Detection (ObjectDetector)")
print("✓ Text Classification (TextPredictor)")
print("✓ Time Series Forecasting (TimeSeriesPredictor)")
print("✓ Multi-Modal Data (MultiModalPredictor)")

print("\n" + "="*80)
print("AutoGluon Demo Complete!")
print("="*80)
print("\nKey Takeaways:")
print("- Best-in-class accuracy through multi-layered ensembling")
print("- Automatic model stacking and bagging")
print("- Support for multiple data types (tabular, image, text)")
print("- Built-in hyperparameter optimization")
print("- Model distillation for faster inference")
print("- Comprehensive leaderboard and model comparison")
print("- Easy deployment with model persistence")


AutoGluon Capabilities Summary:
✓ Tabular Data (TabularPredictor)
✓ Image Classification (ImagePredictor)
✓ Object Detection (ObjectDetector)
✓ Text Classification (TextPredictor)
✓ Time Series Forecasting (TimeSeriesPredictor)
✓ Multi-Modal Data (MultiModalPredictor)

AutoGluon Demo Complete!

Key Takeaways:
- Best-in-class accuracy through multi-layered ensembling
- Automatic model stacking and bagging
- Support for multiple data types (tabular, image, text)
- Built-in hyperparameter optimization
- Model distillation for faster inference
- Comprehensive leaderboard and model comparison
- Easy deployment with model persistence
